# Exploratory Data Analysis and Preprocessing

This notebook performs EDA and data preprocessing for the Heart Disease UCI dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Explore Dataset

We'll use the Heart Disease UCI dataset. Let's load it first.

In [ ]:
# Load dataset
# Note: Download from https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset
# For this example, we'll use a URL or load from file

try:
    # Try loading from file
    df = pd.read_csv('heart.csv')
except FileNotFoundError:
    # If file doesn't exist, download from URL
    import urllib.request
    url = 'https://raw.githubusercontent.com/plotly/datasets/master/heart.csv'
    df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Dataset info
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("\nFirst few rows:")
print(df.head())
print("\n" + "="*50)
print("\nDataset Statistics:")
print(df.describe())

## 2. Check for Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_values.index,
    'Missing Count': missing_values.values,
    'Missing Percentage': missing_percent.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if len(missing_df) > 0:
    print("Missing Values:")
    print(missing_df)
    # Visualize missing values
    plt.figure(figsize=(10, 6))
    sns.barplot(data=missing_df, x='Column', y='Missing Percentage')
    plt.title('Missing Values Percentage')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found!")

## 3. Exploratory Data Analysis

In [ ]:
# Identify target variable (usually the last column or named 'target')
# For Heart Disease UCI, target is typically the last column
target_col = df.columns[-1] if 'target' not in df.columns else 'target'
print(f"Target variable: {target_col}")

# Check class distribution
print("\nClass Distribution:")
print(df[target_col].value_counts())
print(f"\nClass Distribution (%):")
print(df[target_col].value_counts(normalize=True) * 100)

# Visualize class distribution
plt.figure(figsize=(8, 6))
df[target_col].value_counts().plot(kind='bar')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Separate numeric and categorical features
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_features:
    numeric_features.remove(target_col)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

## 4. Data Preprocessing

In [ ]:
# Handle missing values (if any)
# For numeric features, fill with median
for feature in numeric_features:
    if df[feature].isnull().any():
        df[feature].fillna(df[feature].median(), inplace=True)

# For categorical features, fill with mode
for feature in categorical_features:
    if df[feature].isnull().any():
        df[feature].fillna(df[feature].mode()[0], inplace=True)

print("Missing values handled.")

In [ ]:
# Encode categorical variables
label_encoders = {}
df_processed = df.copy()

for feature in categorical_features:
    le = LabelEncoder()
    df_processed[feature] = le.fit_transform(df[feature])
    label_encoders[feature] = le
    print(f"Encoded {feature}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

if len(categorical_features) == 0:
    print("No categorical features to encode.")

In [ ]:
# Prepare features and target
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {list(X.columns)}")

In [ ]:
# Split data: 70% training, 15% validation, 15% test
# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Second split: 50% of temp for validation, 50% for test (15% each of total)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

print(f"\nClass distribution in training: {np.bincount(y_train)}")
print(f"Class distribution in validation: {np.bincount(y_val)}")
print(f"Class distribution in test: {np.bincount(y_test)}")

In [ ]:
# Standardize features (important for k-NN and Naive Bayes)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Features standardized.")
print(f"\nScaled training set statistics:")
print(X_train_scaled.describe())

In [ ]:
# Save preprocessed data for use in main analysis notebook
import pickle

data_dict = {
    'X_train': X_train.values,
    'X_train_scaled': X_train_scaled.values,
    'X_val': X_val.values,
    'X_val_scaled': X_val_scaled.values,
    'X_test': X_test.values,
    'X_test_scaled': X_test_scaled.values,
    'y_train': y_train.values,
    'y_val': y_val.values,
    'y_test': y_test.values,
    'feature_names': list(X.columns),
    'scaler': scaler,
    'label_encoders': label_encoders
}

with open('preprocessed_data.pkl', 'wb') as f:
    pickle.dump(data_dict, f)

print("Preprocessed data saved to 'preprocessed_data.pkl'")
print("\nEDA and Preprocessing Complete!")